# Week 5: Does stability-aware feature selection help a full prediction model?

**Question for the meeting:** If we first keep useful predictors and then penalize predictors that vary across hospitals, does a logistic-regression model perform better at a hospital left out of training?

This is a self-contained continuation of the group's Week 4 work. Last week examined individual feature associations. Here, every reported performance result comes from **fitting a complete multivariable model**.

**Default run:** Midwest source hospitals only; `label_J96.00`; 20, 30, and 50 original features; relationship and distribution penalties; an optional second-outcome check is enabled. **Northeast records are not loaded or used.**

## Run this on your computer
1. Save this notebook anywhere in the team repository. Do not replace last week's notebook.
2. Open it with the `cs197` Jupyter kernel. Keep the authorized full `clean_dataset.parquet` and trusted course `feature_names.pkl` on your own device, not in GitHub.
3. Run from the top. If automatic file search fails, set `DATA_DIR` in the configuration cell to the folder containing those two files.
4. Choose **Restart Kernel and Run All Cells**. Each completed configuration is checkpointed. Rerunning with the same data/settings resumes rather than fitting it again.
5. Send the resulting `week5_meeting_results.zip` and the saved executed notebook to the group. Start the discussion with `meeting_brief.md`.

You only need **this notebook**, not a separate Python module. The ZIP produced by a run contains aggregate tables, feature names, figures, and explanations. It excludes patient records, individual predictions, and local cache files. Review outputs before sharing and use the team's authorized channel.

**Scope:** This is a development experiment, not a certified clinical model or an exact implementation of StableMate. It does not guarantee a positive result. A utility-only rule can legitimately be the best setting.

## 1. Imports and configuration
Use your existing course environment. If an import fails, install only the missing dependency in that environment. For example, a new notebook cell can run `%pip install pyarrow`, followed by a kernel restart.

Dependencies: NumPy, pandas, SciPy, Matplotlib, scikit-learn, pyarrow, and threadpoolctl. The run records installed versions. No downloads, account connections, or network calls are performed by this notebook.

In [ ]:
from pathlib import Path
from dataclasses import dataclass
import hashlib
import importlib.metadata
import json
import math
import pickle
import platform
import re
import time
import warnings
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits
from IPython.display import display

In [ ]:
# Set DATA_DIR only if automatic search cannot identify one processed full dataset.
# Example: DATA_DIR = Path.home() / 'Documents' / 'data'
DATA_DIR = None
OUTPUT_BASE = Path('week5_feature_selection_outputs')

# Default: the full source-hospital development experiment and a second outcome check.
RUN_SECONDARY_OUTCOME = True
RUN_NESTED_LOHO = False          # Optional, substantially more model fits; no Northeast data.
NESTED_OUTER_LIMIT = None        # None = every eligible outer hospital when enabled.
SYNTHETIC_TEST = False           # True ONLY for an explicitly labeled software smoke test.

CFG = {
    'seed': 42,
    'source_region': 'Midwest',
    'primary_outcome': 'label_J96.00',
    'secondary_outcome': 'mortality_at48h',
    'missing_diagnosis_is_absent': True,  # Same convention as the existing group notebook.
    'internal_fraction': 0.20,
    'min_source_rows': 10,               # Fewer rows: exclude and report; never silently lose a hospital.
    'min_loho_rows': 150,                # Counts below refer to the 80% development partition.
    'min_loho_class': 10,                # Both positive and negative outcomes at each omitted hospital.
    'min_loho_hospitals': 5,
    'max_loho_folds': None,              # Set to 3 ONLY for a labeled preview; None = all eligible folds.
    'min_feature_obs': 500,
    'min_feature_observed_rate': 0.05,
    'min_feature_hospital_obs': 30,
    'min_feature_outcome_class': 20,
    'min_score_hospitals': 5,
    'min_relationship_class': 5,
    'min_binary_class': 25,              # At least this many 0s AND 1s in fitting data.
    'min_binary_hospitals': 3,           # At least this many training hospitals contain a 1.
    'max_per_group': 40,
    'max_candidates': 160,
    'minimum_utility': 0.02,             # Exploratory floor: oriented univariate AUROC at least 0.51.
    'utility_pool_multiplier': 2,        # Among top 2*m useful features, apply the stability score.
    'feature_counts': [20, 30, 50],
    'positive_lambdas': [0.5, 1.0],      # Lambda=0 is represented by the utility-only baseline.
    'penalty_metrics': ['relationship', 'distribution'],  # Optional later: add 'missingness'.
    'ks_bins': 64,
    'model_C': 0.10,
    'class_weight': 'balanced',
    'max_iter': 2500,
    'blas_threads': 1,
}

# Keep the scope frozen before examining the new results.
assert 0 < CFG['internal_fraction'] < 1
assert all(int(m) > 0 for m in CFG['feature_counts'])
assert all(lam > 0 for lam in CFG['positive_lambdas'])
assert set(CFG['penalty_metrics']) <= {'relationship', 'distribution', 'missingness'}
assert CFG['penalty_metrics']
print('Primary outcome:', CFG['primary_outcome'])
print('Source-only region:', CFG['source_region'])
print('Selected-feature counts:', CFG['feature_counts'], '| penalties:', [0, *CFG['positive_lambdas']])
print('Northeast evaluation: DISABLED. This notebook does not load Northeast data.')

## 2. What exactly are we measuring?

### Predictive utility
For a feature $j$, use its **observed values** to rank the outcome in the fitting hospitals. If $A_j$ is that single-feature AUROC,

$$U_j = 2\,|A_j - 0.5|.$$

For example, AUROC 0.70 gives utility 0.40; AUROC 0.30 also gives utility 0.40 because the feature is informative in the opposite direction. AUROC 0.50 gives zero. This measures a feature on its own; it does **not** measure its unique contribution after accounting for other predictors.

### Two penalties tested separately
**Relationship instability:** compute a single-feature AUROC at each fitting hospital with at least 30 observed values and 5 cases of each outcome class. Take the population standard deviation of those AUROCs, using at least 5 hospitals. A high value means the feature's association with the label differs across hospitals. We do not reverse directions separately within hospitals before taking the standard deviation.

**Distribution instability:** compare each hospital's observed-value distribution with the pooled distribution at the other fitting hospitals, then average those distances with equal hospital weight. For continuous variables, this is a **64-quantile-bin approximation to the KS distance**. Cut points are fitted again inside every training fold. For binary variables, the distance is the absolute difference in the fraction with value 1. At least 5 sufficiently observed hospitals are required. This is not a KS p-value.

**Missingness instability** is also recorded: the standard deviation of hospitals' observed-value fractions. It is not in the default penalty grid; add `missingness` to `penalty_metrics` before running a separately documented extension.

### Concrete selection rule
1. Apply training-only quality filters and computational group caps. A binary code needs enough 0s, 1s, and hospitals containing a 1; nonmissing zeros alone do not establish useful evidence.
2. The baseline and penalized methods share a candidate pool with all configured instability scores estimable and $U_j\geq 0.02$. This exploratory floor means oriented single-feature AUROC of at least 0.51; it is **not** a significance or clinical threshold.
3. For a budget of $m$ original features, keep the top $2m$ candidates by utility as the useful-feature pool.
4. Rank features inside that pool using

$$S_j = R(U_j)-\lambda R(I_j),$$

where $R$ is the average-tie percentile rank among the shared candidate pool. Select the top $m$. Larger $\lambda$ penalizes instability more. $\lambda=0$ is utility-only selection. The same $m$ and useful-feature pool are used in each paired comparison.

The default grid is **15 configurations**: 3 utility-only budgets plus 2 instability definitions x 2 positive penalties x 3 budgets. There is also an **all-coverage-features control**, for 16 configurations per omitted hospital. That control uses the capped quality-filtered predictors, not every column of the dataset.

We retain a missingness flag for **every selected original predictor**, so 30 original features produce 60 model inputs. Every model uses the same preprocessing and regularized logistic regression.

These rules turn last week's hypothesis into a testable procedure. They are proposed design choices, not findings already established by the previous notebook. The new quality filters differ from the earlier run; do not treat its AUROCs as directly interchangeable with these results.

## 3. Data checks and splitting helpers
These cells define functions; they do not run the experiment yet. Open a collapsed helper cell to inspect the implementation.

The split is approximately 80/20 **inside each included Midwest hospital**. Each included hospital therefore appears on both sides. Repeated records are grouped by patient ID when available; otherwise by hospitalization ID. Hospitals too small to split are explicitly listed as exclusions. A missing longitudinal patient ID limits what can be guaranteed about repeated admissions.

Diagnosis-label missing values follow the previous notebook's explicit absent-code convention. Missing mortality labels are excluded. The number affected is recorded. The outcome remains a recorded post-window label, not automatically a new-onset diagnosis.

In [ ]:
def stable_seed(*parts):
    text = '|'.join(map(str, parts))
    return int(hashlib.sha256(text.encode()).hexdigest()[:8], 16)


def json_default(obj):
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, Path):
        return str(obj)
    raise TypeError(f'Cannot serialize {type(obj)}')


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, default=json_default, allow_nan=False))
    temporary.replace(path)


def save_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(temporary, index=False, float_format='%.17g')
    temporary.replace(path)


def safe_name(value):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value))


def locate_data(data_dir=None):
    if data_dir is not None:
        folders = [Path(data_dir).expanduser().resolve()]
    else:
        cwd = Path.cwd().resolve()
        folders = [cwd, cwd / 'data']
        for base in [cwd, *list(cwd.parents)[:5]]:
            folders += [base / 'data', base / 'cs197_fall2026' / 'data']
        folders += [Path.home() / 'Documents' / 'data',
                    Path.home() / 'Documents' / 'cs197_fall2026' / 'data']
    matches = []
    for folder in dict.fromkeys(p.resolve() for p in folders):
        pair = (folder / 'clean_dataset.parquet', folder / 'feature_names.pkl')
        if all(p.is_file() for p in pair):
            matches.append(pair)
    if len(matches) != 1:
        message = ('No matching data folder found.' if not matches else
                   'More than one processed dataset was found; do not guess which is the full dataset.')
        found = '\n'.join(str(pair[0].parent) for pair in matches)
        raise FileNotFoundError(message + '\nSet DATA_DIR in the configuration cell to the folder '
                                'containing both clean_dataset.parquet and feature_names.pkl.\n' + found)
    return matches[0]


def feature_metadata(feature_sets, available_columns):
    groups, kinds = {}, {}
    for key, kind in [('labs', 'continuous'), ('vitals', 'continuous'),
                      ('icd10_before24h', 'binary'), ('admissiondx', 'binary')]:
        if key not in feature_sets:
            raise KeyError(f'feature_names.pkl is missing the expected key: {key}')
        for name in list(feature_sets[key]):
            name = str(name)
            if name in available_columns and not name.startswith('label_'):
                groups.setdefault(name, key)
                kinds.setdefault(name, kind)
    for name in ['age', 'admissionheight', 'admissionweight']:
        if name in available_columns:
            groups[name] = 'demographic'
            kinds[name] = 'continuous'
    for name in list(groups):
        if name in {'hospitalid', 'hospital_region', 'hospital_numbedscategory',
                    'patienthealthsystemstayid', 'patientunitstayid', 'uniquepid',
                    'mortality_at48h'}:
            del groups[name]
            del kinds[name]
    if not groups:
        raise ValueError('No allowed first-window predictors were found in the data.')
    return pd.DataFrame({'feature': list(groups),
                         'feature_group': [groups[x] for x in groups],
                         'kind': [kinds[x] for x in groups]})


def load_midwest(data_dir, cfg):
    try:
        import pyarrow.parquet as pq
    except ImportError as exc:
        raise ImportError('The data loader needs pyarrow. In a NEW notebook cell run '
                          '`%pip install pyarrow`, then restart the kernel.') from exc
    data_path, names_path = locate_data(data_dir)
    # This pickle must be the trusted metadata file produced by the course preprocessing.
    with names_path.open('rb') as handle:
        feature_sets = pickle.load(handle)
    columns = set(pq.ParquetFile(data_path).schema_arrow.names)
    required = {'hospitalid', 'hospital_region', cfg['primary_outcome']}
    absent = required - columns
    if absent:
        raise KeyError(f'Processed data missing required columns: {sorted(absent)}')
    meta = feature_metadata(feature_sets, columns)
    requested = [cfg['primary_outcome'], cfg['secondary_outcome']]
    identifiers = [c for c in ['patienthealthsystemstayid', 'uniquepid'] if c in columns]
    read_columns = sorted(set(meta.feature) | {'hospitalid', 'hospital_region'} |
                          (set(requested) & columns) | set(identifiers))
    # Parquet row filtering: Northeast values and labels are not loaded for this experiment.
    data = pd.read_parquet(data_path, columns=read_columns,
                           filters=[('hospital_region', '==', cfg['source_region'])], engine='pyarrow')
    if data.empty:
        raise ValueError(f'No {cfg["source_region"]} rows found. Check the hospital_region values.')
    if data.hospitalid.isna().any():
        raise ValueError('Source data contain missing hospital IDs.')
    if 'patienthealthsystemstayid' in data and data.patienthealthsystemstayid.duplicated().any():
        raise ValueError('Expected one row per patienthealthsystemstayid. Recheck Week 1 aggregation.')
    # Keep a real patient ID only for split protection. Never export these IDs.
    if 'uniquepid' in data and data.uniquepid.notna().all():
        unit_col = 'uniquepid'
        spanning = data.groupby(unit_col).hospitalid.nunique()
        spanning = spanning.index[spanning > 1]
        removed = int(data[unit_col].isin(spanning).sum())
        data = data.loc[~data[unit_col].isin(spanning)].copy()
        unit_description = 'patient (uniquepid); patients spanning source hospitals excluded'
    elif 'patienthealthsystemstayid' in data:
        unit_col = 'patienthealthsystemstayid'
        removed = 0
        unit_description = 'hospitalization; no complete longitudinal patient ID available'
    else:
        unit_col = None
        removed = 0
        unit_description = 'row; no patient/hospitalization ID available, independence unverified'
        warnings.warn('No patient identifier: repeated-patient leakage cannot be ruled out.')
    if unit_col is None:
        data['_unit'] = [f'row_{i}' for i in range(len(data))]
    else:
        if data[unit_col].isna().any():
            raise ValueError(f'{unit_col} contains missing IDs; fix before splitting.')
        data['_unit'] = data[unit_col].astype(str)
    data['hospitalid'] = data.hospitalid.astype(str)
    data = data.reset_index(drop=True)
    audit_rows = []
    for feature in meta.feature:
        original = data[feature]
        if feature == 'age':
            original = original.astype('string').str.strip().replace({'> 89': '90', '>89': '90'})
        numeric = pd.to_numeric(original, errors='coerce').astype(float)
        numeric = numeric.replace([np.inf, -np.inf], np.nan)
        audit_rows.append({'feature': feature,
                           'new_missing_after_numeric_conversion': int((original.notna() & numeric.isna()).sum())})
        data[feature] = numeric
    for feature in meta.loc[meta.kind.eq('binary'), 'feature']:
        vals = data[feature].dropna()
        if not vals.isin([0., 1.]).all():
            raise ValueError(f'{feature} is listed as a binary predictor but contains nonbinary values.')
    # Fingerprint the actual source data and metadata, not just its filename.
    digest = hashlib.sha256()
    digest.update(pd.util.hash_pandas_object(data, index=True).values.tobytes())
    digest.update(meta.to_csv(index=False).encode())
    manifest = {'dataset_filename': data_path.name, 'source_region': cfg['source_region'],
                'source_rows_loaded': len(data), 'source_hospitals_loaded': data.hospitalid.nunique(),
                'source_fingerprint': digest.hexdigest(), 'split_unit': unit_description,
                'cross_hospital_patient_rows_excluded': removed,
                'northeast_loaded_or_used': False}
    return data, meta, manifest, pd.DataFrame(audit_rows)


def prepare_outcome(data, meta, outcome, cfg):
    if outcome not in data:
        raise KeyError(f'Outcome {outcome} is not present.')
    frame = data.copy()
    raw = frame[outcome]
    y = pd.to_numeric(raw, errors='coerce')
    invalid = raw.notna() & y.isna()
    if invalid.any() or not y.dropna().isin([0, 1]).all():
        raise ValueError(f'{outcome} must contain only 0, 1, or missing.')
    n_missing = int(y.isna().sum())
    if outcome.startswith('label_') and cfg['missing_diagnosis_is_absent']:
        y = y.fillna(0)
        policy = 'missing diagnosis label treated as absent, following the previous notebook'
    else:
        frame = frame.loc[y.notna()].copy()
        y = y.loc[y.notna()]
        policy = 'rows with missing outcomes excluded'
    frame['_y'] = y.astype(int)
    # Exclude all labels and any pre-window column from the outcome's ICD family.
    names = meta.copy()
    excluded = []
    if outcome.startswith('label_'):
        family = outcome[len('label_'):].split('.')[0].upper()
        excluded = names.loc[names.feature.str.upper().str.contains(family, regex=False), 'feature'].tolist()
        names = names.loc[~names.feature.isin(excluded)].copy()
    return frame.reset_index(drop=True), names, {
        'outcome': outcome, 'missing_labels_before_policy': n_missing,
        'missing_label_policy': policy, 'excluded_outcome_family_features': excluded,
        'label_note': 'A recorded post-window diagnosis is not proof of new disease onset.'}


def within_hospital_split(frame, cfg):
    train_parts, audit_parts, rows = [], [], []
    for hospital, part in frame.groupby('hospitalid', sort=True):
        if len(part) < cfg['min_source_rows'] or part['_unit'].nunique() < 2:
            rows.append({'hospitalid': hospital, 'status': 'excluded_too_small_to_split',
                         'n_all': len(part), 'n_development': 0, 'n_internal': 0,
                         'positive_development': 0, 'positive_internal': 0})
            continue
        seed = stable_seed(cfg['seed'], 'internal', hospital)
        if part['_unit'].duplicated().any():
            splitter = GroupShuffleSplit(n_splits=1, test_size=cfg['internal_fraction'], random_state=seed)
            a, b = next(splitter.split(part, groups=part['_unit']))
            dev, internal = part.iloc[a], part.iloc[b]
        else:
            counts = part['_y'].value_counts()
            n_test = math.ceil(len(part) * cfg['internal_fraction'])
            stratify = part['_y'] if (len(counts) == 2 and counts.min() >= 2 and
                                      min(n_test, len(part) - n_test) >= 2) else None
            dev, internal = train_test_split(part, test_size=cfg['internal_fraction'],
                                             random_state=seed, stratify=stratify)
        if set(dev['_unit']) & set(internal['_unit']):
            raise AssertionError('Patient/hospitalization overlap across the internal split.')
        train_parts.append(dev)
        audit_parts.append(internal)
        rows.append({'hospitalid': hospital, 'status': 'included', 'n_all': len(part),
                     'n_development': len(dev), 'n_internal': len(internal),
                     'positive_development': int(dev['_y'].sum()),
                     'positive_internal': int(internal['_y'].sum())})
    if not train_parts:
        raise ValueError('No source hospitals can be split. Check whether this is the demo data.')
    dev = pd.concat(train_parts).sort_index()
    internal = pd.concat(audit_parts).sort_index()
    assert set(dev.hospitalid) == set(internal.hospitalid)
    assert not (set(dev['_unit']) & set(internal['_unit']))
    return dev, internal, pd.DataFrame(rows)


def evaluation_hospitals(dev, cfg):
    stats = dev.groupby('hospitalid')['_y'].agg(n='size', positives='sum')
    stats['negatives'] = stats.n - stats.positives
    stats['eligible_for_loho'] = ((stats.n >= cfg['min_loho_rows']) &
                                  (stats.positives >= cfg['min_loho_class']) &
                                  (stats.negatives >= cfg['min_loho_class']))
    return stats

## 4. Training-only feature scoring
Every omitted-hospital fold rebuilds the candidate filter, utility gate, ranks, and distribution cut points from the fitting hospitals only. Insufficient instability evidence leads to exclusion from the shared ranking pool, not a falsely favorable stability score.

In [ ]:
def auc_from_feature(x, y):
    observed = np.isfinite(x)
    x, y = x[observed], y[observed]
    n1, n0 = int(y.sum()), int(len(y) - y.sum())
    if n1 == 0 or n0 == 0:
        return np.nan
    ranks = rankdata(x, method='average')
    return float((ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def rank01(values):
    return pd.Series(values).rank(method='average', pct=True)


def score_features(train, meta, cfg):
    """No omitted-hospital rows are accepted by this function."""
    y = train['_y'].to_numpy(dtype=int)
    hospitals, hcode = np.unique(train.hospitalid.to_numpy(), return_inverse=True)
    nh = len(hospitals)
    hn = np.bincount(hcode, minlength=nh)
    hrows = [np.flatnonzero(hcode == i) for i in range(nh)]
    coverage, exclusions = [], []
    for row in meta.itertuples(index=False):
        x = train[row.feature].to_numpy(dtype=float)
        ok = np.isfinite(x)
        nobs = np.bincount(hcode[ok], minlength=nh)
        positive_obs = int(y[ok].sum())
        negative_obs = int(ok.sum() - positive_obs)
        enough = int((nobs >= cfg['min_feature_hospital_obs']).sum())
        reason = None
        if (ok.sum() < cfg['min_feature_obs'] or ok.mean() < cfg['min_feature_observed_rate']):
            reason = 'too_few_observed_values'
        elif enough < cfg['min_score_hospitals']:
            reason = 'insufficient_hospital_coverage'
        elif min(positive_obs, negative_obs) < cfg['min_feature_outcome_class']:
            reason = 'too_few_observed_outcome_cases'
        elif np.unique(x[ok]).size < 2:
            reason = 'constant'
        elif row.kind == 'binary':
            present = int((x[ok] == 1).sum())
            absent = int((x[ok] == 0).sum())
            present_hospitals = len(np.unique(hcode[ok & (x == 1)]))
            if min(present, absent) < cfg['min_binary_class'] or present_hospitals < cfg['min_binary_hospitals']:
                reason = 'binary_feature_too_rare'
        if reason:
            exclusions.append({'feature': row.feature, 'reason': reason})
        else:
            coverage.append({'feature': row.feature, 'feature_group': row.feature_group, 'kind': row.kind,
                             'nonmissing': int(ok.sum()), 'observed_rate': float(ok.mean()),
                             'hospital_coverage': enough})
    coverage = pd.DataFrame(coverage)
    if coverage.empty:
        raise ValueError('No predictors pass the training-only quality filters.')
    ranked = coverage.sort_values(['hospital_coverage', 'nonmissing', 'feature'], ascending=[False, False, True])
    capped = ranked.groupby('feature_group', sort=False).head(cfg['max_per_group'])
    capped = capped.head(cfg['max_candidates']).copy()
    for f in set(coverage.feature) - set(capped.feature):
        exclusions.append({'feature': f, 'reason': 'predeclared_computational_cap'})
    rows = []
    for row in capped.itertuples(index=False):
        x = train[row.feature].to_numpy(dtype=float)
        ok = np.isfinite(x)
        nobs = np.bincount(hcode[ok], minlength=nh).astype(float)
        miss_sd = float(np.std(nobs / hn, ddof=0))
        auc = auc_from_feature(x, y)
        rel_auc = []
        for indices in hrows:
            observed = indices[np.isfinite(x[indices])]
            yh = y[observed]
            if (len(observed) >= cfg['min_feature_hospital_obs'] and
                min(int(yh.sum()), int(len(yh) - yh.sum())) >= cfg['min_relationship_class']):
                rel_auc.append(auc_from_feature(x[observed], yh))
        relationship = (float(np.std(rel_auc, ddof=0))
                        if len(rel_auc) >= cfg['min_score_hospitals'] else np.nan)
        use = (nobs >= cfg['min_feature_hospital_obs']) & ((nobs.sum() - nobs) >= cfg['min_feature_hospital_obs'])
        if row.kind == 'binary':
            totals = np.bincount(hcode[ok], weights=x[ok], minlength=nh)
            ph = totals[use] / nobs[use]
            prest = (totals.sum() - totals[use]) / (nobs.sum() - nobs[use])
            distances = np.abs(ph - prest)
        else:
            # All cut points are recomputed from THIS training fold, never the omitted hospital.
            edges = np.unique(np.quantile(x[ok], np.linspace(0, 1, cfg['ks_bins'] + 1)[1:-1]))
            bins = np.searchsorted(edges, x[ok], side='right')
            b = len(edges) + 1
            hist = np.bincount(hcode[ok] * b + bins, minlength=nh*b).reshape(nh, b)
            cdf_h = hist[use].cumsum(axis=1) / nobs[use, None]
            rest = hist.sum(axis=0)[None, :] - hist[use]
            cdf_rest = rest.cumsum(axis=1) / (nobs.sum() - nobs[use])[:, None]
            distances = np.abs(cdf_h - cdf_rest).max(axis=1)
        distribution = (float(np.mean(distances))
                        if use.sum() >= cfg['min_score_hospitals'] else np.nan)
        rows.append({**row._asdict(), 'single_feature_auroc': auc, 'utility': 2*abs(auc - .5),
                     'missingness': miss_sd, 'relationship': relationship,
                     'distribution': distribution, 'relationship_hospitals': len(rel_auc),
                     'distribution_hospitals': int(use.sum())})
    scores = pd.DataFrame(rows)
    shared = scores.utility.ge(cfg['minimum_utility']) & scores[list(cfg['penalty_metrics'])].notna().all(axis=1)
    scores['eligible_for_ranking'] = shared
    for column in ['utility', 'relationship', 'distribution', 'missingness']:
        scores[column + '_rank'] = np.nan
        # Rank within the SAME common candidate pool for baseline and penalized methods.
        scores.loc[shared, column + '_rank'] = scores.loc[shared, column].rank(pct=True, method='average')
    scores['ranking_exclusion'] = np.where(shared, '', np.where(
        scores.utility.lt(cfg['minimum_utility']), 'below_utility_floor', 'insufficient_instability_evidence'))
    return scores, pd.DataFrame(exclusions, columns=['feature', 'reason'])


def make_specs(cfg):
    specs = [{'id': 'all_coverage', 'metric': 'all', 'm': 0, 'lambda': 0.0}]
    for m in cfg['feature_counts']:
        specs.append({'id': f'utility_m{m}', 'metric': 'utility', 'm': int(m), 'lambda': 0.0})
        for metric in cfg['penalty_metrics']:
            for lam in cfg['positive_lambdas']:
                tag = str(lam).replace('.', 'p')
                specs.append({'id': f'{metric}_l{tag}_m{m}', 'metric': metric,
                              'm': int(m), 'lambda': float(lam)})
    return specs


def select_features(scores, spec, cfg):
    if spec['metric'] == 'all':
        return scores.feature.tolist()
    pool = scores.loc[scores.eligible_for_ranking].sort_values(['utility', 'feature'], ascending=[False, True]).copy()
    m = int(spec['m'])
    if len(pool) < m:
        raise ValueError(f'Only {len(pool)} rankable predictors; {m} were requested. No silent resizing.')
    pool = pool.head(min(len(pool), cfg['utility_pool_multiplier'] * m))
    penalty = 0 if spec['metric'] == 'utility' else pool[spec['metric'] + '_rank']
    pool['score'] = pool.utility_rank - spec['lambda'] * penalty
    return pool.sort_values(['score', 'utility', 'feature'], ascending=[False, False, True]).head(m).feature.tolist()

## 5. Full-model LOHO and checkpointing
For an omitted Midwest hospital H:

- **Fit:** use the 80% development rows from the other source hospitals to select features, fit medians/scales, and train logistic regression.
- **Omitted-hospital evaluation:** predict H's development rows. No rows from H are used for training or feature ranking.
- **Internal diagnostic:** evaluate the same model on the reserved 20% rows from the remaining hospitals. These rows are not used for feature selection or to choose the grid setting.

The primary result is **AUROC on the omitted hospital**. The gap is internal pooled AUROC minus omitted-hospital AUROC. A smaller gap alone is not evidence of a better model, since internal performance may simply have fallen.

The grid is selected by **mean omitted-hospital AUROC**, giving each eligible hospital equal weight. A configuration must complete and converge at all evaluated hospitals before it can be selected. A requested feature count is never silently reduced. Unexpected errors stop the run; saved earlier results are retained.

The default LOHO grid is for **hyperparameter selection**. Reporting its best mean as an unbiased improvement would be optimistic. An optional nested check below separates hyperparameter choice from each outer test hospital.

In [ ]:
def transform_values(train_x, *evaluation_x):
    """Training medians/scales only; one missingness flag per original selected predictor."""
    medians = np.nanmedian(train_x, axis=0)
    if not np.isfinite(medians).all():
        raise ValueError('A selected feature has no finite training values.')
    scaler = StandardScaler()
    ztrain = scaler.fit_transform(np.where(np.isnan(train_x), medians, train_x))
    result = [np.column_stack([ztrain, np.isnan(train_x).astype(float)])]
    for x in evaluation_x:
        z = scaler.transform(np.where(np.isnan(x), medians, x))
        result.append(np.column_stack([z, np.isnan(x).astype(float)]))
    return result


def performance_metrics(y, p):
    y = np.asarray(y, dtype=int)
    if len(np.unique(y)) < 2:
        return {'n': len(y), 'positives': int(y.sum()), 'prevalence': float(y.mean()) if len(y) else None,
                'auroc': None, 'average_precision': None, 'brier': None}
    return {'n': len(y), 'positives': int(y.sum()), 'prevalence': float(y.mean()),
            'auroc': float(roc_auc_score(y, p)),
            'average_precision': float(average_precision_score(y, p)),
            'brier': float(brier_score_loss(y, p))}


def fit_model(train, test, internal, features, cfg):
    frames = [train, test, internal]
    matrices = [d[features].to_numpy(dtype=float) for d in frames]
    a, b, c = transform_values(*matrices)
    model = LogisticRegression(C=cfg['model_C'], solver='lbfgs',
                               class_weight=cfg['class_weight'], max_iter=cfg['max_iter'],
                               random_state=cfg['seed'])
    with warnings.catch_warnings(record=True) as caught, threadpool_limits(limits=cfg['blas_threads']):
        warnings.simplefilter('always', ConvergenceWarning)
        model.fit(a, train['_y'].to_numpy(dtype=int))
    nonconverged = any(issubclass(w.category, ConvergenceWarning) for w in caught)
    outer = performance_metrics(test['_y'], model.predict_proba(b)[:, 1])
    inside = performance_metrics(internal['_y'], model.predict_proba(c)[:, 1])
    row = {'n_features': len(features), 'n_model_columns': 2*len(features),
           'train_n': len(train), 'train_hospitals': train.hospitalid.nunique(),
           'test_n': outer['n'], 'test_positives': outer['positives'], 'test_prevalence': outer['prevalence'],
           'heldout_auroc': outer['auroc'], 'heldout_average_precision': outer['average_precision'],
           'heldout_brier': outer['brier'], 'internal_auroc': inside['auroc'],
           'internal_n': inside['n'], 'converged': not nonconverged,
           'iterations': int(model.n_iter_.max())}
    row['gap_auroc'] = (inside['auroc'] - outer['auroc']) if inside['auroc'] is not None and outer['auroc'] is not None else None
    return row


def run_one_fold(dev, internal, train_ids, test_id, specs, meta, cfg, cache_dir):
    train_ids = sorted(set(map(str, train_ids)))
    test_id = str(test_id)
    if test_id in train_ids:
        raise AssertionError('Omitted hospital was included in training.')
    train = dev.loc[dev.hospitalid.isin(train_ids)]
    test = dev.loc[dev.hospitalid.eq(test_id)]
    inside = internal.loc[internal.hospitalid.isin(train_ids)]
    if len(test) == 0 or len(inside) == 0 or train['_y'].nunique() < 2:
        raise ValueError('Empty evaluation set or single-class training data.')
    assert not (set(train['_unit']) & set(test['_unit']))
    assert not (set(train['_unit']) & set(inside['_unit']))
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    fold_meta_path = cache_dir / 'fold.json'
    fold_info = {'training_hospitals': train_ids, 'omitted_hospital': test_id}
    if fold_meta_path.exists() and json.loads(fold_meta_path.read_text()) != fold_info:
        raise RuntimeError('Fold cache does not match training hospitals. Use a fresh output directory.')
    write_json(fold_meta_path, fold_info)
    score_path = cache_dir / 'scores.csv'
    if score_path.exists():
        scores = pd.read_csv(score_path, float_precision='round_trip')
    else:
        scores, exclusions = score_features(train, meta, cfg)
        save_csv(scores, score_path)
        save_csv(exclusions, cache_dir / 'quality_exclusions.csv')
    rows = []
    for spec in specs:
        path = cache_dir / f'{spec["id"]}.json'
        if path.exists():
            saved = json.loads(path.read_text())
            if saved.get('spec') != spec:
                raise RuntimeError('Cached configuration mismatch.')
            rows.append(saved['result'])
            continue
        row = {'fold_hospital': test_id, 'config': spec['id'], 'metric': spec['metric'],
               'm': spec['m'], 'lambda': spec['lambda']}
        try:
            features = select_features(scores, spec, cfg)
        except ValueError as exc:
            row.update(status='skipped', error=str(exc), n_features=0, converged=False)
            write_json(path, {'spec': spec, 'result': row, 'selected_features': []})
            rows.append(row)
            continue
        # Stop on unexpected fitting errors instead of dropping difficult hospitals silently.
        result = fit_model(train, test, inside, features, cfg)
        row.update(result, status='ok', error='')
        write_json(path, {'spec': spec, 'result': row, 'selected_features': features})
        rows.append(row)
    return rows


def compare_results(results, expected_folds, specs):
    records = []
    for spec in specs:
        rows = results.loc[results.config.eq(spec['id'])]
        ok = rows.loc[rows.status.eq('ok')].copy()
        if not ok.empty:
            ok = ok.loc[ok.heldout_auroc.notna()]
        complete = set(ok.fold_hospital.astype(str)) == set(map(str, expected_folds))
        record = {**spec, 'completed_hospitals': len(ok), 'expected_hospitals': len(expected_folds),
                  'complete': complete, 'all_converged': bool(ok.converged.all()) if len(ok) else False}
        for column in ['heldout_auroc', 'gap_auroc', 'internal_auroc', 'heldout_average_precision']:
            record['mean_' + column] = float(ok[column].mean()) if len(ok) else np.nan
            record['median_' + column] = float(ok[column].median()) if len(ok) else np.nan
        record['worst_heldout_auroc'] = float(ok.heldout_auroc.min()) if len(ok) else np.nan
        record['sd_heldout_auroc'] = float(ok.heldout_auroc.std(ddof=1)) if len(ok) > 1 else np.nan
        records.append(record)
    return pd.DataFrame(records).sort_values(['mean_heldout_auroc', 'm', 'lambda', 'id'],
                                              ascending=[False, True, True, True], na_position='last')


def choose_rule(summary, specs, family='proposed'):
    # Equal hospital weights, not patient weights; reject incomplete/nonconverged settings.
    usable = summary.loc[summary.complete & summary.all_converged & summary.metric.ne('all')].copy()
    if family == 'utility':
        usable = usable.loc[usable.metric.eq('utility')]
    elif family == 'penalized':
        usable = usable.loc[usable.metric.ne('utility')]
    if usable.empty:
        return None
    chosen = usable.sort_values(['mean_heldout_auroc', 'm', 'lambda', 'metric'],
                                 ascending=[False, True, True, True]).iloc[0]['id']
    return next(s.copy() for s in specs if s['id'] == chosen)


def paired_comparison(results, candidate_id, baseline_id):
    ok = results.loc[results.status.eq('ok')]
    left = ok.loc[ok.config.eq(candidate_id), ['fold_hospital', 'heldout_auroc', 'gap_auroc', 'test_n', 'test_positives']]
    right = ok.loc[ok.config.eq(baseline_id), ['fold_hospital', 'heldout_auroc', 'gap_auroc']]
    pair = left.merge(right, on='fold_hospital', suffixes=('_candidate', '_baseline'), validate='one_to_one')
    pair['delta_auroc'] = pair.heldout_auroc_candidate - pair.heldout_auroc_baseline
    return pair


def run_source_loho(source_data, meta, outcome, cfg, root, specs=None, phase='development'):
    frame, ometa, label_info = prepare_outcome(source_data, meta, outcome, cfg)
    dev, internal, audit = within_hospital_split(frame, cfg)
    out = Path(root) / safe_name(outcome) / phase
    out.mkdir(parents=True, exist_ok=True)
    write_json(out / 'label_policy.json', label_info)
    save_csv(audit, out / 'source_split_audit.csv')
    estats = evaluation_hospitals(dev, cfg)
    save_csv(estats.reset_index(), out / 'hospital_eligibility.csv')
    fold_ids = sorted(estats.index[estats.eligible_for_loho].tolist())
    if len(fold_ids) < cfg['min_loho_hospitals']:
        raise ValueError(f'{outcome}: only {len(fold_ids)} hospitals qualify for LOHO. '
                         'Do not lower thresholds merely to make a demo run look like full data.')
    complete_fold_ids = fold_ids.copy()
    if cfg['max_loho_folds'] is not None and len(fold_ids) > cfg['max_loho_folds']:
        fold_ids = sorted(np.random.default_rng(cfg['seed']).choice(
            fold_ids, size=cfg['max_loho_folds'], replace=False).tolist())
    preview = len(fold_ids) != len(complete_fold_ids)
    specs = make_specs(cfg) if specs is None else specs
    hospital_ids = sorted(dev.hospitalid.unique().tolist())
    print(f'\n{outcome}: {len(dev):,} development rows; {len(internal):,} internal rows.')
    print(f'{len(hospital_ids)} hospitals in BOTH partitions. LOHO: {len(fold_ids)}/{len(complete_fold_ids)} eligible hospitals.')
    print(f'{len(specs)} configurations per fold. Northeast is not loaded or evaluated.')
    if preview:
        print('PREVIEW ONLY: this subset is a pipeline check, not a completed Week 5 comparison.')
    results = []
    for i, h in enumerate(fold_ids, start=1):
        print(f'  Fold {i}/{len(fold_ids)}: omit H{h}', flush=True)
        rows = run_one_fold(dev, internal, [x for x in hospital_ids if x != h], h,
                            specs, ometa, cfg, out / '_cache' / f'hospital_{safe_name(h)}')
        results.extend(rows)
        save_csv(pd.DataFrame(results), out / 'model_loho_results.csv')
    results = pd.DataFrame(results)
    summary = compare_results(results, fold_ids, specs)
    save_csv(summary, out / 'configuration_summary.csv')
    rules = {name: choose_rule(summary, specs, name) for name in ['utility', 'penalized', 'proposed']}
    write_json(out / 'development_choices.json', {
        'preview': preview, 'choices': rules, 'selection_objective': 'mean held-out source-hospital AUROC',
        'warning': 'These settings were selected on the same LOHO results; not independent confirmation.'})
    return {'outcome': outcome, 'dev': dev, 'internal': internal, 'meta': ometa, 'out': out,
            'fold_ids': fold_ids, 'preview': preview, 'results': results, 'summary': summary,
            'rules': rules, 'specs': specs, 'audit': audit, 'label_info': label_info}

## 6. Plotting and interpretation helpers
Every plot is saved as a separate PNG with a plain-language caption. No unexplained shortfall scores, p-values, or confidence intervals are generated. Source LOHO folds overlap in their training data; their differences are shown descriptively rather than treated as independent statistical replicates.

In [ ]:
def record_figure(fig, directory, filename, caption):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(directory / filename, dpi=180, bbox_inches='tight')
    (directory / (Path(filename).stem + '_caption.txt')).write_text(caption + '\n')
    plt.show()
    plt.close(fig)


def make_performance_figures(run, cfg):
    summary, out, results = run['summary'], run['out'], run['results']
    good = summary.loc[summary.complete & summary.all_converged]
    for metric in cfg['penalty_metrics']:
        fig, ax = plt.subplots(figsize=(8, 5))
        plotted = False
        for m in cfg['feature_counts']:
            rows = good.loc[good.m.eq(m) & good.metric.isin(['utility', metric])].sort_values('lambda')
            if len(rows):
                ax.plot(rows['lambda'], rows.mean_heldout_auroc, marker='o', label=f'{m} features')
                plotted = True
        if not plotted:
            plt.close(fig)
            continue
        ax.set(xlabel='Stability penalty (lambda)', ylabel='Mean omitted-hospital AUROC',
               title=f'{run["outcome"]}: {metric} penalty\nSource-hospital tuning results, not an independent test')
        ax.legend(title='Original features; same logistic regression')
        record_figure(fig, out, f'loho_performance_{metric}.png',
            f'Each point is the equally weighted mean AUROC across {len(run["fold_ids"])} omitted Midwest hospitals. '
            'Features, medians, scales, and models are rebuilt without the omitted hospital. '
            'Lambda=0 selects by utility only. These curves are used to choose settings and are development results; '
            'they are not an unbiased estimate of the best setting\'s future performance.')
    chosen = run['rules']['penalized']
    if chosen is None:
        return
    matched_id = f'utility_m{chosen["m"]}'
    if matched_id not in set(results.config):
        return
    pair = paired_comparison(results, chosen['id'], matched_id).sort_values('fold_hospital')
    save_csv(pair, out / 'paired_auroc_vs_same_size_utility.csv')
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(pair))
    ax.scatter(x, pair.delta_auroc)
    ax.axhline(0, linewidth=1, linestyle='--')
    ax.set_xticks(x, pair.fold_hospital, rotation=70)
    ax.set(xlabel='Omitted Midwest hospital', ylabel='AUROC change: penalized minus utility-only',
           title=f'{chosen["metric"]}, lambda={chosen["lambda"]}: {chosen["m"]} features in both models')
    record_figure(fig, out, 'paired_hospital_auroc_changes.png',
        'Each point compares two models at the same omitted source hospital. Positive means the penalized model '
        'had higher AUROC. This comparison uses the best penalized configuration from the tuning grid, so it is '
        'exploratory. No p-value or confidence interval is shown; training sets overlap across folds.')


def inspect_features(run, cfg):
    out = run['out']
    full_dir = out / 'feature_inspection'
    full_dir.mkdir(exist_ok=True)
    # Descriptive fit on ALL source development rows, after validation, never on Northeast.
    scores, excluded = score_features(run['dev'], run['meta'], cfg)
    save_csv(scores, full_dir / 'source_feature_scores.csv')
    save_csv(excluded, full_dir / 'source_quality_exclusions.csv')
    chosen = run['rules']['penalized'] or run['rules']['utility']
    if chosen is None:
        print('No complete, converged configuration. Inspect configuration_summary.csv before interpreting features.')
        return
    metric = chosen['metric'] if chosen['metric'] != 'utility' else cfg['penalty_metrics'][0]
    m = chosen['m']
    lam_chosen = chosen['lambda']
    selection_rows = []
    for lam in [0., *cfg['positive_lambdas']]:
        spec = {'id': 'inspection', 'metric': 'utility' if lam == 0 else metric, 'm': m, 'lambda': lam}
        try:
            selected = select_features(scores, spec, cfg)
        except ValueError:
            continue
        for rank, f in enumerate(selected, 1):
            meta = scores.set_index('feature').loc[f]
            selection_rows.append({'lambda': lam, 'm': m, 'rank': rank, 'feature': f,
                                   'feature_group': meta.feature_group, 'utility': meta.utility,
                                   'instability': meta[metric], 'score': meta.utility_rank - lam*meta[metric+'_rank']})
    selections = pd.DataFrame(selection_rows)
    save_csv(selections, full_dir / 'selected_features_across_lambda.csv')
    valid = scores.loc[scores.eligible_for_ranking].copy()
    valid['score'] = valid.utility_rank - lam_chosen * valid[metric + '_rank']
    chosen_set = set(select_features(scores, chosen, cfg))
    utility_set = set(select_features(scores, {'metric': 'utility', 'm': m, 'lambda': 0.}, cfg))
    valid['selected'] = valid.feature.isin(chosen_set)
    valid['utility_only_selected'] = valid.feature.isin(utility_set)
    valid['selection_change'] = np.select([
        valid.feature.isin(chosen_set & utility_set),
        valid.feature.isin(utility_set - chosen_set),
        valid.feature.isin(chosen_set - utility_set)], ['kept_by_both', 'removed_by_penalty', 'added_by_penalty'], default='not_in_either_top_set')
    valid['team_clinical_interpretation'] = ''
    review = valid.loc[valid.selected | valid.utility_only_selected].sort_values(['selection_change', 'utility'], ascending=[True, False])
    save_csv(review, full_dir / 'features_to_discuss.csv')
    fig, ax = plt.subplots(figsize=(9, 5.5))
    for group, part in valid.groupby('feature_group'):
        ax.scatter(part[metric + '_rank'], part.utility, alpha=.7, label=group)
    for row in review.loc[review.selection_change.eq('removed_by_penalty')].head(6).itertuples():
        ax.annotate(row.feature, (getattr(row, metric + '_rank'), row.utility), fontsize=8)
    ax.set(xlabel=f'{metric.capitalize()} instability percentile (larger = less consistent)',
           ylabel='Single-feature predictive utility: 2 x |AUROC - 0.5|',
           title='Which useful features does the penalty remove?')
    ax.legend(fontsize=8)
    record_figure(fig, full_dir, 'utility_and_instability.png',
        f'Each point is a rankable predictor scored on the full source 80% development partition. '
        f'Labels show up to six predictors removed from the utility-only top {m} when the penalty is added. '
        'This is a description of the selection rule, not a test that the individual feature causes prediction failure.')
    fig, ax = plt.subplots(figsize=(9, 5))
    groups = list(valid.groupby('feature_group', sort=True))
    ax.boxplot([g.score.to_numpy() for _, g in groups], showmeans=True)
    ax.set_xticks(np.arange(1, len(groups)+1), [f'{name}\n(n={len(g)})' for name, g in groups], rotation=20)
    ax.set(ylabel=f'Selection score: utility rank - {lam_chosen:g} x {metric} rank',
           title='Distribution of feature scores within each feature group')
    record_figure(fig, full_dir, 'feature_scores_by_group.png',
        'Boxes summarize scores of rankable source predictors in each group; group counts are shown. '
        'Higher scores rank ahead of lower scores within the utility-gated candidate pool. '
        'Binary and continuous predictors can have different measurement properties; this is not a causal group comparison.')
    counts = selections.groupby(['lambda', 'feature_group']).size().unstack(fill_value=0)
    fig, ax = plt.subplots(figsize=(9, 5))
    counts.plot.bar(ax=ax)
    ax.set(xlabel='Stability penalty (lambda)', ylabel='Number of selected original features',
           title=f'How the {m}-feature set changes as the {metric} penalty increases')
    ax.tick_params(axis='x', rotation=0)
    record_figure(fig, full_dir, 'selected_feature_types_by_lambda.png',
        f'All bars at one penalty value sum to {m}. The outcome, model, source development data, feature count, '
        f'and utility gate are fixed; only the {metric} penalty changes. '
        'Exact feature names and additions/removals are saved in the accompanying CSVs.')
    # Inclusion across independently rebuilt LOHO training folds, for ONE fixed configuration.
    frequency, eligible_count = {}, {}
    fold_count = 0
    for h in run['fold_ids']:
        directory = out / '_cache' / f'hospital_{safe_name(h)}'
        result_file = directory / (chosen['id'] + '.json')
        if not result_file.exists():
            continue
        record = json.loads(result_file.read_text())
        if record['result']['status'] != 'ok':
            continue
        fold_count += 1
        s = pd.read_csv(directory / 'scores.csv')
        for f in s.loc[s.eligible_for_ranking, 'feature']:
            eligible_count[f] = eligible_count.get(f, 0) + 1
        for f in record['selected_features']:
            frequency[f] = frequency.get(f, 0) + 1
    frequency_rows = []
    for f in sorted(set(eligible_count) | set(frequency)):
        denom = eligible_count.get(f, 0)
        frequency_rows.append({'feature': f, 'selected_folds': frequency.get(f, 0),
                               'eligible_folds': denom, 'completed_folds': fold_count,
                               'selection_fraction_all_folds': frequency.get(f, 0)/max(1, fold_count),
                               'selection_fraction_when_eligible': frequency.get(f, 0)/denom if denom else np.nan})
    freq = pd.DataFrame(frequency_rows)
    if not freq.empty:
        freq = freq.sort_values(['selected_folds', 'feature'], ascending=[False, True])
        save_csv(freq, full_dir / 'selection_frequency_across_loho.csv')
        fig, ax = plt.subplots(figsize=(9, 6))
        top = freq.head(15).sort_values('selection_fraction_all_folds')
        ax.barh(top.feature, top.selection_fraction_all_folds)
        ax.set(xlim=(0, 1), xlabel='Fraction of completed LOHO folds selecting this feature',
               title='Features most consistently kept by the fixed penalized rule')
        record_figure(fig, full_dir, 'frequently_selected_features.png',
            'For the displayed fixed rule, feature scores and quality filters were recalculated in every fold. '
            'The denominator is all completed folds, including folds where a feature failed eligibility. '
            'This is a stability diagnostic, not a statistical test or clinical endorsement.')
    print('\nFeatures for discussion:')
    display(review[['feature', 'feature_group', 'utility', metric, 'selection_change']].head(25))

In [ ]:
def nested_source_loho(primary, cfg, outer_limit=None):
    if primary['preview']:
        raise ValueError('Complete the development run before enabling nested validation.')
    dev, internal, meta = primary['dev'], primary['internal'], primary['meta']
    all_h = sorted(dev.hospitalid.unique())
    outer_ids = primary['fold_ids'].copy()
    if outer_limit is not None:
        outer_ids = sorted(np.random.default_rng(cfg['seed']).choice(
            outer_ids, min(outer_limit, len(outer_ids)), replace=False).tolist())
    out = primary['out'].parent / 'nested_validation'
    out.mkdir(exist_ok=True)
    final_rows, choice_rows = [], []
    for index, h in enumerate(outer_ids, 1):
        print(f'Nested outer hospital {index}/{len(outer_ids)}: H{h}', flush=True)
        allowed = [a for a in all_h if a != h]
        inner_ids = [a for a in primary['fold_ids'] if a != h]
        if len(inner_ids) < 3:
            raise ValueError('Too few inner hospitals for nested selection.')
        inner_results = []
        for j in inner_ids:
            inner_results.extend(run_one_fold(dev, internal, [a for a in allowed if a != j], j,
                primary['specs'], meta, cfg, out / '_cache' / f'outer_{safe_name(h)}' / f'inner_{safe_name(j)}'))
        inner_summary = compare_results(pd.DataFrame(inner_results), inner_ids, primary['specs'])
        baseline = choose_rule(inner_summary, primary['specs'], 'utility')
        proposed = choose_rule(inner_summary, primary['specs'], 'proposed')
        if baseline is None or proposed is None:
            raise ValueError(f'No complete/converged inner rule for H{h}. Inspect the inner results.')
        specs = list({s['id']: s for s in [baseline, proposed]}.values())
        outer_rows = run_one_fold(dev, internal, allowed, h, specs, meta, cfg,
                                  out / '_cache' / f'outer_{safe_name(h)}' / 'outer_evaluation')
        mapping = {r['config']: r for r in outer_rows}
        for role, chosen in [('utility_only', baseline), ('proposed_procedure', proposed)]:
            final_rows.append({**mapping[chosen['id']], 'role': role})
            choice_rows.append({'fold_hospital': h, 'role': role, **chosen})
        save_csv(pd.DataFrame(final_rows), out / 'nested_outer_results.csv')
        save_csv(pd.DataFrame(choice_rows), out / 'inner_selected_settings.csv')
    final = pd.DataFrame(final_rows)
    a = final.loc[final.role.eq('proposed_procedure'), ['fold_hospital', 'heldout_auroc']]
    b = final.loc[final.role.eq('utility_only'), ['fold_hospital', 'heldout_auroc']]
    pair = a.merge(b, on='fold_hospital', suffixes=('_proposed', '_utility'), validate='one_to_one')
    pair['delta_auroc'] = pair.heldout_auroc_proposed - pair.heldout_auroc_utility
    save_csv(pair, out / 'nested_paired_changes.csv')
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(np.arange(len(pair)), pair.delta_auroc)
    ax.axhline(0, linestyle='--', linewidth=1)
    ax.set_xticks(np.arange(len(pair)), pair.fold_hospital, rotation=65)
    ax.set(ylabel='AUROC change: tuned procedure minus tuned utility baseline', xlabel='Outer omitted hospital',
           title='Nested source-only validation: settings chosen without the outer hospital')
    record_figure(fig, out, 'nested_paired_changes.png',
        'Both the penalty rule and the baseline feature count were chosen using inner LOHO that excluded '
        'the outer hospital. Each point is a paired model comparison at that outer hospital. '
        'Outer training sets overlap, so these points are not independent replicated experiments.')
    return {'paired': pair, 'out': out, 'all_outer_folds_completed': len(outer_ids) == len(primary['fold_ids'])}

In [ ]:
def rule_description(spec):
    if spec is None:
        return 'No complete, converged configuration'
    if spec['metric'] == 'utility':
        return f'utility only, {spec["m"]} features'
    return f'{spec["metric"]} penalty, lambda={spec["lambda"]:g}, {spec["m"]} features'


def meeting_notes(primary, secondary, nested, cfg, manifest, root):
    rules, summary = primary['rules'], primary['summary']
    lines = [
        '# Week 5 meeting notes', '',
        '## What we tested',
        'Does penalizing inconsistent predictors improve a full logistic-regression model compared with selecting predictors by usefulness alone?',
        f'Outcome: {primary["outcome"]}. Source region: {cfg["source_region"]}. Northeast data were not loaded or used.',
        f'{len(primary["dev"]):,} rows in source development and {len(primary["internal"]):,} in internal evaluation; '
        f'{primary["dev"].hospitalid.nunique()} hospitals appear in both.',
        f'Independent splitting unit: {manifest["split_unit"]}.',
        f'{len(primary["fold_ids"])} omitted source hospitals were evaluated. Preview-only run: {primary["preview"]}.', '',
        '## What the quantities mean',
        'Utility = 2 x |single-feature AUROC - 0.5|, calculated on observed training values. It is not a multivariate coefficient or causal effect.',
        'Relationship instability = standard deviation of the single-feature AUROC across training hospitals with enough observations and cases.',
        f'Distribution instability = mean hospital-versus-other-training-hospitals CDF distance, approximated using {cfg["ks_bins"]} training-only quantile bins; for binary features it is an exact prevalence difference.',
        f'All selectors share a quality-filtered pool, a utility floor of {cfg["minimum_utility"]:g}, and adequate evidence for the configured instability measures.',
        f'For m selected features, we first keep the top {cfg["utility_pool_multiplier"]} x m by utility, then rank utility percentile minus lambda times instability percentile.',
        'Each selected original feature supplies its value and a missingness indicator to the same model.', '',
        '## Development results (used for choosing settings)',
        f'Best utility-only setting: {rule_description(rules["utility"])}.',
        f'Best penalized setting: {rule_description(rules["penalized"])}.',
        f'Best setting including lambda=0: {rule_description(rules["proposed"])}.',
    ]
    if rules['penalized'] is not None:
        candidate = rules['penalized']
        baseline_id = f'utility_m{candidate["m"]}'
        pair = paired_comparison(primary['results'], candidate['id'], baseline_id)
        if len(pair):
            gain = pair.delta_auroc
            lines += [f'At the same feature count, penalized minus utility-only mean omitted-hospital AUROC = {gain.mean():+.4f}; '
                      f'median = {gain.median():+.4f}.',
                      f'It improved AUROC at {int((gain > 1e-10).sum())}/{len(gain)} hospitals, '
                      f'was worse at {int((gain < -1e-10).sum())}, and tied at {int((gain.abs() <= 1e-10).sum())}.',
                      'These are observed tuning-set differences, not proof that the selected penalty will generalize better.']
    if rules['utility'] and rules['proposed']:
        su = summary.set_index('id')
        gain_best = su.loc[rules['proposed']['id'], 'mean_heldout_auroc'] - su.loc[rules['utility']['id'], 'mean_heldout_auroc']
        lines.append(f'Compared with the separately tuned utility-only baseline, the best grid setting changes mean AUROC by {gain_best:+.4f}.')
        if rules['proposed']['metric'] == 'utility':
            lines.append('The tuning grid did not select a positive stability penalty. That is a valid negative result; do not describe it as a stability-aware win.')
    lines += ['', '## Second outcome check']
    if secondary is None:
        lines.append('Not completed. Do not claim that the method was tested on a second outcome.')
    else:
        candidate = rules['proposed']
        baseline_id = f'utility_m{candidate["m"]}' if candidate else None
        if candidate and candidate['id'] in set(secondary['results'].config):
            pair = paired_comparison(secondary['results'], candidate['id'], baseline_id)
            lines += [f'Outcome: {secondary["outcome"]}, {len(secondary["fold_ids"])} LOHO hospitals.',
                      'The primary-outcome settings were held fixed; they were not retuned for the second outcome.',
                      f'Mean paired AUROC change versus same-size utility-only = {pair.delta_auroc.mean():+.4f}.' if len(pair) else
                      'No complete paired results. Inspect the eligibility/skipped-configuration table.']
    lines += ['', '## Independent tuning/evaluation check']
    if nested is None:
        lines.append('Nested source-only validation was not run. The best grid results are selection results, not independent evidence of improvement.')
    else:
        gain = nested['paired'].delta_auroc
        lines += [f'Nested outer comparisons: {len(gain)}. All eligible outer hospitals completed: {nested["all_outer_folds_completed"]}.',
                  f'Mean paired outer AUROC change = {gain.mean():+.4f}; wins = {int((gain > 1e-10).sum())}/{len(gain)}.',
                  'The omitted outer hospital was excluded from feature ranking and hyperparameter choice.']
    lines += ['', '## What to discuss with Arthi',
              '1. Which setting should move forward, and is the observed improvement large enough to matter?',
              '2. Open feature_inspection/features_to_discuss.csv: name two retained and two removed predictors. Add clinical interpretations only after checking what those variables mean.',
              '3. Does the second outcome show the same direction, and how did the eligible hospital cohort change?',
              '4. Use the performance curve, paired hospital plot, and feature-selection changes with their saved captions. Define every axis and every denominator in the report.',
              '5. Do not reuse the old feature-shortfall terminology for these model AUROCs. No formal confidence intervals or significance claims are generated here.', '',
              '## Limits to state',
              'Utility is univariate; correlated predictors may be individually useful but redundant in a full model.',
              'Eligibility and computational caps define the candidate pool. Results are not a comparison over every possible EHR feature.',
              'The internal reference uses the same remaining hospitals and different patients; a smaller gap alone can result from lower internal performance.',
              'The low-utility floor is an exploratory choice, not a clinical or statistical significance threshold.',
              'Source LOHO folds share training data. Hospital win counts and dispersion are descriptive, not independent-test p-values.',
              'Earlier Northeast results have already been viewed; that region is not a pristine final confirmation dataset.',
              'If a longitudinal patient ID is absent, different admissions of the same person cannot be reliably separated.',
              'A later recorded diagnosis may reflect documentation timing rather than new disease onset.',
              'Balanced class weights can change calibration; average precision and Brier score are secondary diagnostics.', '',
              '## File map',
              'configuration_summary.csv: every setting, mean AUROC, gap, convergence and completion counts.',
              'model_loho_results.csv: one model/configuration/omitted-hospital result per row.',
              'feature_inspection/: exact scores, additions/removals, selection frequency, and readable figures.',
              '_cache/: local resumable aggregate checkpoints. Not needed for the meeting handoff.',
              'No patient-level records or individual predictions are exported.', '']
    text = '\n'.join(lines)
    (Path(root) / 'meeting_brief.md').write_text(text)
    print(text)
    return text


def create_share_bundle(root):
    root = Path(root)
    dest = root / 'week5_meeting_results.zip'
    with zipfile.ZipFile(dest, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in root.rglob('*'):
            if not p.is_file() or '_cache' in p.parts or p == dest:
                continue
            if p.suffix.lower() in {'.csv', '.json', '.md', '.txt', '.png'}:
                z.write(p, p.relative_to(root))
    print('Aggregate meeting bundle:', dest.resolve())
    return dest

In [ ]:
def synthetic_source(cfg):
    """Small synthetic hospitals for software tests only. No clinical records."""
    rng = np.random.default_rng(817)
    rows = []
    for h in range(8):
        n = 240 + 10*h
        latent = rng.normal(size=n)
        shift = (h-3.5)/3.5
        p = 1/(1+np.exp(-(-0.9 + .9*latent + .25*shift)))
        y = rng.binomial(1, p)
        mortality = rng.binomial(1, 1/(1+np.exp(-(-1.3+.6*latent))))
        d = {'hospitalid': np.repeat(str(h), n), 'hospital_region': np.repeat('Midwest', n),
             '_unit': [f'SYNTHETIC_{h}_{i}' for i in range(n)],
             'label_J96.00': y, 'mortality_at48h': mortality,
             'age': 60+12*latent+rng.normal(0,3,n),
             'admissionheight': rng.normal(170, 8, n), 'admissionweight': rng.normal(75,12,n)}
        for j in range(10):
            x = (.3+.1*(j%3))*latent + rng.normal(0, 1, n) + shift*(j%4)/3
            x[rng.random(n) < .05 + .015*(h+j)%0.2] = np.nan
            d[f'lab_{j}'] = x
        for j in range(8):
            x = (.8 if j < 4 else shift)*latent + rng.normal(size=n)
            x[rng.random(n) < .1] = np.nan
            d[f'vital_{j}'] = x
        for j in range(8):
            d[f'dx_{j}'] = rng.binomial(1, 1/(1+np.exp(-(-.4+.3*latent+.1*j))))
            d[f'admission_{j}'] = rng.binomial(1, .25+.02*j, n)
        d['J96.00'] = y.astype(float)  # Must be excluded from the primary task.
        d['lab_constant'] = np.ones(n)
        d['lab_empty'] = np.full(n, np.nan)
        d['dx_rare'] = np.zeros(n)
        if h == 0:
            d['dx_rare'][0] = 1
        rows.append(pd.DataFrame(d))
    frame = pd.concat(rows, ignore_index=True)
    names = {'labs': [f'lab_{j}' for j in range(10)] + ['lab_constant','lab_empty'],
             'vitals': [f'vital_{j}' for j in range(8)],
             'icd10_before24h': [f'dx_{j}' for j in range(8)] + ['J96.00','dx_rare'],
             'admissiondx': [f'admission_{j}' for j in range(8)]}
    meta = feature_metadata(names, set(frame))
    settings = dict(cfg)
    settings.update(min_source_rows=10, min_loho_rows=80, min_loho_class=10,
                    min_loho_hospitals=5, min_feature_obs=80,
                    min_feature_hospital_obs=15, min_feature_outcome_class=10,
                    min_score_hospitals=3, min_relationship_class=3,
                    min_binary_class=10, min_binary_hospitals=2,
                    feature_counts=[5, 10, 15], minimum_utility=0.0, ks_bins=32)
    fingerprint = hashlib.sha256(pd.util.hash_pandas_object(frame, index=True).values.tobytes()).hexdigest()
    manifest = {'dataset_filename': 'SYNTHETIC_MEMORY_ONLY', 'source_region': 'Midwest',
                'source_rows_loaded': len(frame), 'source_hospitals_loaded': 8,
                'source_fingerprint': fingerprint, 'split_unit': 'synthetic independent rows',
                'cross_hospital_patient_rows_excluded': 0, 'northeast_loaded_or_used': False,
                'synthetic_test_not_research': True}
    return frame, meta, manifest, pd.DataFrame(), settings

## 7. Load source data and create a reproducible output folder
No raw rows are displayed. Automatic path search refuses to guess when multiple processed datasets are present.

`SYNTHETIC_TEST=True` runs small artificial hospitals only to test the software. Its output is prominently labeled and must never be described as eICU findings. Leave it **False** for the group experiment.

In [ ]:
ENGINE_FINGERPRINT = 'bb3ec4e02f0a9af801699794afd9507f395b5761b7adec1bb02bb1de89bc5b10'
if SYNTHETIC_TEST:
    print('SYNTHETIC SOFTWARE TEST ONLY. THESE ARE NOT CLINICAL RESULTS.')
    source_data, predictor_meta, data_manifest, conversion_audit, CFG = synthetic_source(CFG)
else:
    source_data, predictor_meta, data_manifest, conversion_audit = load_midwest(DATA_DIR, CFG)

versions = {}
for package in ['numpy', 'pandas', 'scipy', 'matplotlib', 'scikit-learn', 'pyarrow', 'threadpoolctl']:
    try:
        versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        versions[package] = 'not installed'
versions['python'] = platform.python_version()
# Include the actual helper bytecode in the cache key, so edits cannot silently reuse old scores.
from types import CodeType
def _stable_code_signature(code_object):
    constants = [(_stable_code_signature(x) if isinstance(x, CodeType) else repr(x))
                 for x in code_object.co_consts]
    return [code_object.co_code.hex(), list(code_object.co_names),
            list(code_object.co_varnames), code_object.co_argcount, constants]
_engine_functions = [stable_seed, feature_metadata, load_midwest, prepare_outcome,
    within_hospital_split, evaluation_hospitals, auc_from_feature, rank01, score_features,
    make_specs, select_features, transform_values, performance_metrics, fit_model,
    run_one_fold, compare_results, choose_rule, paired_comparison, run_source_loho,
    nested_source_loho]
_actual_code = [[f.__name__, _stable_code_signature(f.__code__), repr(f.__defaults__)]
                for f in _engine_functions]
ENGINE_FINGERPRINT = hashlib.sha256((ENGINE_FINGERPRINT + json.dumps(_actual_code)).encode()).hexdigest()
# A preview limit does not change training or scores; completed folds can be reused later.
fingerprint_cfg = {k: v for k, v in CFG.items() if k != 'max_loho_folds'}
run_key = hashlib.sha256(json.dumps({
    'engine': ENGINE_FINGERPRINT, 'config': fingerprint_cfg,
    'source_data': data_manifest['source_fingerprint'], 'versions': versions,
}, sort_keys=True).encode()).hexdigest()[:12]
prefix = 'SYNTHETIC_TEST_' if SYNTHETIC_TEST else 'midwest_'
RUN_DIR = OUTPUT_BASE / (prefix + run_key)
RUN_DIR.mkdir(parents=True, exist_ok=True)
write_json(RUN_DIR / 'run_manifest.json', {'config': CFG, 'data': data_manifest,
    'software_versions': versions, 'engine_fingerprint': ENGINE_FINGERPRINT,
    'secondary_enabled': RUN_SECONDARY_OUTCOME, 'nested_enabled': RUN_NESTED_LOHO})
if not conversion_audit.empty:
    save_csv(conversion_audit, RUN_DIR / 'numeric_conversion_audit.csv')
print('Source rows:', f'{len(source_data):,}', '| source hospitals:', source_data.hospitalid.nunique())
print('Prediction unit:', data_manifest['split_unit'])
print('Output folder:', RUN_DIR.resolve())
print('Versions:', versions)

## 8. Run the primary development experiment
This is the main compute cell. Progress is printed once per omitted hospital. Completed configuration results are saved immediately, even if a later cell fails or the kernel is interrupted.

Default: all eligible Midwest folds. A preview limit of 3 is only for checking setup; return it to `None` for the complete experiment. Do not choose a subset based on how well the model performs.

In [ ]:
primary = run_source_loho(source_data, predictor_meta, CFG['primary_outcome'], CFG, RUN_DIR)
cols = ['id', 'completed_hospitals', 'complete', 'all_converged',
        'mean_heldout_auroc', 'median_heldout_auroc', 'worst_heldout_auroc', 'mean_gap_auroc']
display(primary['summary'][cols].round(4))
print('Development choices:')
for family, rule in primary['rules'].items():
    print(f'  {family}: {rule_description(rule)}')
if not primary['summary']['all_converged'].all():
    print('Some configurations were skipped or did not converge. Review the table before reporting a comparison.')

## 9. Does adding a penalty improve full-model prediction?
Read the performance curves across lambda and m, then the **paired AUROC change** at each omitted hospital. Positive values mean a penalized model beat a utility-only model with the same number of original features. These are tuning comparisons, not causal effects of individual predictors.

In [ ]:
make_performance_figures(primary, CFG)

## 10. Which features are retained or removed?
The first table lists features to discuss with the group. It leaves a column for your own clinical interpretation instead of inventing one.

The figures show scores by feature group, selection changes as lambda increases, and how often each feature is retained across LOHO folds. The full-source ranking is descriptive: model evaluation above still used independently rebuilt rankings in every fold.

In [ ]:
inspect_features(primary, CFG)

## 11. Second outcome: keep the primary settings fixed
By default this repeats full-model source LOHO for `mortality_at48h`. The chosen primary-outcome rule and a same-size utility-only baseline are held fixed; feature scores and predictors are re-estimated for mortality within each fold. This is an exploratory cross-outcome check, not another hyperparameter search.

The eligible hospitals can differ between outcomes. A changed population is recorded and should be mentioned when comparing results. If the primary grid selects lambda=0, the frozen rule and utility baseline are identical; a zero difference is expected, not a software error.

In [ ]:
secondary = None
if RUN_SECONDARY_OUTCOME and primary['rules']['proposed'] is not None:
    frozen = primary['rules']['proposed'].copy()
    same_size_baseline = {'id': f'utility_m{frozen["m"]}', 'metric': 'utility',
                          'm': frozen['m'], 'lambda': 0.0}
    frozen_specs = list({s['id']: s for s in [same_size_baseline, frozen]}.values())
    if CFG['secondary_outcome'] in source_data:
        secondary = run_source_loho(source_data, predictor_meta, CFG['secondary_outcome'], CFG,
                                     RUN_DIR, specs=frozen_specs, phase='fixed_primary_settings')
        display(secondary['summary'][['id', 'completed_hospitals', 'complete',
                  'mean_heldout_auroc', 'mean_gap_auroc']].round(4))
        pair = paired_comparison(secondary['results'], frozen['id'], same_size_baseline['id'])
        save_csv(pair, secondary['out'] / 'paired_frozen_rule_vs_utility.csv')
        if len(pair):
            fig, ax = plt.subplots(figsize=(9, 5))
            ax.scatter(np.arange(len(pair)), pair.delta_auroc)
            ax.axhline(0, linestyle='--', linewidth=1)
            ax.set_xticks(np.arange(len(pair)), pair.fold_hospital, rotation=65)
            ax.set(xlabel='Omitted Midwest hospital', ylabel='AUROC change: frozen rule minus utility-only',
                   title=f'{CFG["secondary_outcome"]}: primary settings transferred without retuning')
            record_figure(fig, secondary['out'], 'second_outcome_paired_changes.png',
                'Each point compares the frozen primary rule against utility-only selection with the same feature count '
                'at the same source hospital for the second outcome. Outcome-specific eligibility and rankings are recomputed. '
                'This is an exploratory robustness check, not an untouched external test.')
    else:
        print('Secondary outcome absent. Record this as not completed, rather than substituting a new outcome automatically.')
else:
    print('Second-outcome run skipped; do not claim it is complete.')

## 12. Optional nested validation: a stronger test before external evaluation
**Off by default before the meeting.** Set `RUN_NESTED_LOHO=True` in the configuration and run this cell to enable it. It fits many more models and also checkpoints its work.

For each outer omitted Midwest hospital, the notebook repeats inner source-hospital LOHO without that hospital to choose the penalty and feature count. It separately tunes the utility-only baseline's feature count. Both selected models are then evaluated at the outer omitted hospital. The procedure may choose lambda=0.

This addresses the bias from choosing settings and reporting their performance on the same folds. It is still source-region validation, and outer folds share training data. No automatic formal significance claim is made. A limited outer preview must not be described as a complete nested analysis.

In [ ]:
nested = None
if RUN_NESTED_LOHO:
    nested = nested_source_loho(primary, CFG, outer_limit=NESTED_OUTER_LIMIT)
    display(nested['paired'].round(4))
else:
    print('Nested validation not run. Default results are for choosing settings, not final confirmation.')

## 13. Meeting handoff
Start with `meeting_brief.md`. It states what ran, the exact definitions, which settings were selected, what changed, and the limits of the evidence. Do not copy it into the progress report without reviewing it.

The handoff ZIP contains aggregate results and readable PNGs with captions. It contains no raw eICU rows, patient identifiers, individual predictions, or fitted model objects. Share only through the authorized team/course channel. Keep the raw data and `_cache` local.

In [ ]:
meeting_notes(primary, secondary, nested, CFG, data_manifest, RUN_DIR)
if SYNTHETIC_TEST:
    brief = RUN_DIR / 'meeting_brief.md'
    brief.write_text('# SYNTHETIC SOFTWARE TEST — NOT EICU RESULTS\n\n' + brief.read_text())
BUNDLE_PATH = create_share_bundle(RUN_DIR)
print('DONE. Save the notebook, then share the aggregate ZIP shown above.')

## 14. Before writing the report

**Keep the central claim simple.** “We trained the same model after choosing features in different ways and compared its AUROC at hospitals excluded from training.” Then state the feature score and actual result.

Use **model-level AUROC change**, not last week's undefined “feature shortfall,” for this experiment. Distinguish (a) choosing a setting, (b) evaluating a tuning procedure with nested folds, and (c) later testing on an external region. If no penalty helps, say so.

A smaller internal-to-external gap is not enough: check absolute omitted-hospital AUROC, paired wins/losses, and whether performance fell at the most difficult hospitals. Keep clinical interpretations of removed features as hypotheses, not causes established by this experiment.

The original Northeast test results have already been viewed during earlier exploration. This notebook deliberately avoids them. Further validation should use a frozen plan and clearly disclose which data have already influenced decisions.

### Implementation references
The experimental question, outcome names, feature families, regional split, and 80/20 setup come from the group's uploaded notebooks and the Week 5 instructions. The utility gate, explicit data-quality checks, bounded grid, checkpoint format, and nested-validation option are additions in this notebook.

- scikit-learn, **Common pitfalls and recommended practices**, especially fitting preprocessing and feature selection only on training data: https://scikit-learn.org/stable/common_pitfalls.html
- scikit-learn, **Nested versus non-nested cross-validation**, on separating parameter selection from evaluation: https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html
- scikit-learn, **LogisticRegression**, for the fixed prediction model: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

No result from a synthetic smoke test should be submitted as a clinical-data result. No claim is made that this new notebook reproduces the earlier notebooks' exact sample sizes or AUROCs.